In [4]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [8]:
!pip install rouge_score


  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=b8f9fa45c3a09a2ac811d6376270f0888f22d53766641748f5ccd4413939abf2
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [9]:
!pip install bert_score


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.9 MB/s eta 0:00:00


In [17]:

!pip install transformers evaluate rouge_score


from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
import evaluate


test_data = [
    {
        "complex_text": "The agreement is agreed upon by both the parties.",
        "simplified_text": "Both parties have agreed to the contract.",
        "punjabi_translation": "ਦੋਹਾਂ ਪੱਖਾਂ ਨੇ ਸਮਝੌਤੇ 'ਤੇ ਸਹਿਮਤੀ ਦਿੱਤੀ ਹੈ।",
        "malayalam_translation": "എരുത്ത് പാര്‍ട്ടികളും കരാറിന് സമ്മതിച്ചു."
    },
    {
        "complex_text": "Your data will only be used for service improvement and will not be shared with other parties.",
        "simplified_text": "We will only use your data to improve our service and not share it with anyone else.",
        "punjabi_translation": "ਤੁਹਾਡੇ ਡੇਟਾ ਨੂੰ ਸਿਰਫ ਸੇਵਾ ਸੁਧਾਰ ਲਈ ਵਰਤਿਆ ਜਾਵੇਗਾ ਅਤੇ ਕਿਸੇ ਹੋਰ ਨਾਲ ਸਾਂਝਾ ਨਹੀਂ ਕੀਤਾ ਜਾਵੇਗਾ।",
        "malayalam_translation": "നിങ്ങളുടെ ഡാറ്റ സേവന മെച്ചപ്പെടുത്തലിന് മാത്രമേ ഉപയോഗിക്കുകയുള്ളൂ, മറ്റാരുമായും പങ്കിടാറില്ല."
    },
    {
        "complex_text": "The company is not liable for damages arising from misuse of this product.",
        "simplified_text": "The company is not responsible for harm caused by misuse of this product.",
        "punjabi_translation": "ਇਸ ਉਤਪਾਦ ਦੇ ਗਲਤ ਉਪਯੋਗ ਤੋਂ ਪੈਦਾ ਹੋਏ ਨੁਕਸਾਨ ਲਈ ਕੰਪਨੀ ਜ਼ਿੰਮੇਵਾਰ ਨਹੀਂ ਹੈ।",
        "malayalam_translation": "ഈ ഉൽപ്പന്നം തെറ്റായി ഉപയോഗിച്ചതിനാൽ സംഭവിക്കുന്ന നാശത്തിന് കമ്പനി ഉത്തരവാദിയല്ല."
    },
    {
        "complex_text": "The information provided is for educational purposes only and does not constitute legal advice.",
        "simplified_text": "The information is only for learning and is not legal advice.",
        "punjabi_translation": "ਦਿੱਤੀ ਜਾਣਕਾਰੀ ਸਿਰਫ ਸਿੱਖਣ ਲਈ ਹੈ ਅਤੇ ਕਾਨੂੰਨੀ ਸਲਾਹ ਨਹੀਂ ਹੈ।",
        "malayalam_translation": "നൽകിയ വിവരങ്ങൾ പഠനത്തിന് മാത്രമാണ്, നിയമപരമായ ഉപദേശം അല്ല."
    }
]


simplify_model_path = "/content/drive/MyDrive/legal_simplifier/final_model"
punjabi_model_path = "/content/drive/MyDrive/punjabi_model"
malayalam_model_path = "/content/drive/MyDrive/en-ml-finetuned-60k"


simplify_tokenizer = AutoTokenizer.from_pretrained(simplify_model_path, local_files_only=True)
simplify_model = AutoModelForSeq2SeqLM.from_pretrained(simplify_model_path, local_files_only=True)
simplifier_pipeline = pipeline(
    "text2text-generation",
    model=simplify_model,
    tokenizer=simplify_tokenizer
)


punjabi_tokenizer = AutoTokenizer.from_pretrained(punjabi_model_path, local_files_only=True)
punjabi_model = AutoModelForSeq2SeqLM.from_pretrained(punjabi_model_path, local_files_only=True)
punjabi_pipeline = pipeline(
    "translation",
    model=punjabi_model,
    tokenizer=punjabi_tokenizer,
    src_lang="en",
    tgt_lang="pa"
)


ml_tokenizer = AutoTokenizer.from_pretrained(malayalam_model_path, local_files_only=True)
ml_model = AutoModelForSeq2SeqLM.from_pretrained(malayalam_model_path, local_files_only=True)
ml_pipeline = pipeline(
    "translation",
    model=ml_model,
    tokenizer=ml_tokenizer,
    src_lang="en",
    tgt_lang="ml"
)


simplified_outputs = []
punjabi_outputs = []
malayalam_outputs = []

for item in test_data:
    # Simplify English
    simp = simplifier_pipeline(item["complex_text"], max_new_tokens=200)[0]["generated_text"]
    simplified_outputs.append(simp)

    # Translate to Punjabi
    punj = punjabi_pipeline(simp, max_new_tokens=200)[0]["translation_text"]
    punjabi_outputs.append(punj)

    # Translate to Malayalam
    ml = ml_pipeline(simp, max_new_tokens=200)[0]["translation_text"]
    malayalam_outputs.append(ml)


bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

# Simplification evaluation
simpl_ref = [item["simplified_text"] for item in test_data]
bleu_simp = bleu.compute(predictions=simplified_outputs, references=simpl_ref)
rouge_simp = rouge.compute(predictions=simplified_outputs, references=simpl_ref)
bertscore_simp = bertscore.compute(predictions=simplified_outputs, references=simpl_ref, lang="en")

# Punjabi evaluation
punj_ref = [item["punjabi_translation"] for item in test_data]
bleu_punj = bleu.compute(predictions=punjabi_outputs, references=punj_ref)
rouge_punj = rouge.compute(predictions=punjabi_outputs, references=punj_ref)
bertscore_punj = bertscore.compute(predictions=punjabi_outputs, references=punj_ref, lang="pa")

# Malayalam evaluation
ml_ref = [item["malayalam_translation"] for item in test_data]
bleu_ml = bleu.compute(predictions=malayalam_outputs, references=ml_ref)
rouge_ml = rouge.compute(predictions=malayalam_outputs, references=ml_ref)
bertscore_ml = bertscore.compute(predictions=malayalam_outputs, references=ml_ref, lang="ml")


print("===== Simplifier Metrics =====")
print("BLEU:", bleu_simp)
print("ROUGE:", rouge_simp)
print("BERTScore:", bertscore_simp)

print("\n===== Punjabi Translator Metrics =====")
print("BLEU:", bleu_punj)
print("ROUGE:", rouge_punj)
print("BERTScore:", bertscore_punj)

print("\n===== Malayalam Translator Metrics =====")
print("BLEU:", bleu_ml)
print("ROUGE:", rouge_ml)
print("BERTScore:", bertscore_ml)




Device set to use cpu
Device set to use cpu
Device set to use cpu
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


===== Simplifier Metrics =====
BLEU: {'bleu': 0.17111349920478516, 'precisions': [0.4909090909090909, 0.19607843137254902, 0.1276595744680851, 0.06976744186046512], 'brevity_penalty': 1.0, 'length_ratio': 1.0576923076923077, 'translation_length': 55, 'reference_length': 52}
ROUGE: {'rouge1': np.float64(0.4507239819004525), 'rouge2': np.float64(0.1779891304347826), 'rougeL': np.float64(0.40131221719457016), 'rougeLsum': np.float64(0.40470588235294125)}
BERTScore: {'precision': [0.768892228603363, 0.9431266188621521, 0.965794026851654, 0.9340130090713501], 'recall': [0.8530471324920654, 0.9410120844841003, 0.9661344885826111, 0.9522354602813721], 'f1': [0.8087864518165588, 0.9420681595802307, 0.9659642577171326, 0.9430361986160278], 'hashcode': 'roberta-large_L17_no-idf_version=0.3.12(hug_trans=4.57.0)'}

===== Punjabi Translator Metrics =====
BLEU: {'bleu': 0.12421291084010563, 'precisions': [0.4897959183673469, 0.26666666666666666, 0.07317073170731707, 0.02702702702702703], 'brevity_pe